In [1]:
import torch
from sonics import HFAudioClassifier

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model = HFAudioClassifier.from_pretrained("awsaf49/sonics-spectttra-alpha-120s")
model.to(device)
model.eval()

HFAudioClassifier(
  (ft_extractor): FeatureExtractor(
    (audio2melspec): MelSpectrogram(
      (spectrogram): Spectrogram()
      (mel_scale): MelScale()
    )
    (amplitude_to_db): AmplitudeToDB()
    (normalizer): MeanStdNorm()
  )
  (augment): AugmentLayer(
    (mixup): MixUp(num_classes=1, p=0.5, alpha=2.5, inplace=True)
    (time_freq_mask): SpecAugment()
  )
  (encoder): SpecTTTra(
    (st_tokenizer): STTokenizer(
      (temporal_tokenizer): Tokenizer1D(
        (conv1d): Conv1d(128, 384, kernel_size=(3,), stride=(3,), bias=False)
        (act): GELU(approximate='none')
        (pos_encoder): LearnedPositionalEncoding()
        (norm_pre): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      )
      (spectral_tokenizer): Tokenizer1D(
        (conv1d): Conv1d(3744, 384, kernel_size=(1,), stride=(1,), bias=False)
        (act): GELU(approximate='none')
        (pos_encoder): LearnedPositionalEncoding()
        (norm_pre): LayerNorm((384,), eps=1e-06, elementwise_affine=T

In [2]:
import os
import glob
import soxr
import numpy as np
from torch.utils.data import Dataset
from src.utils import load_audio
from src.data import resample, pitch_shift


def get_sonics_file_paths(data_dir, music_generator, split):
    assert music_generator in ["suno_v5", "suno_v3_5", "udio_v120"], "Unsupported music generator in Sonics dataset"
    assert split in ["train", "test"], "Unsupported split in Sonics dataset"
    with open(f"{data_dir}/{music_generator}_{split}.txt", "r") as f:
        file_paths = f.read().splitlines()
    return file_paths


class SonicsDataset(Dataset):
    def __init__(
            self,
            data_dir: str,
            model_name: str,
            max_duration: int = None,
            sampling_rate: int = 16000,
            attack_type: str = "noattack",
            attack_range: tuple = None,
            mode: str = "continuous",
            max_files_per_class: int = 1000,
        ):
        assert model_name in ["suno_v3_5", "udio_v30", "udio_v120"], "Unsupported music model"
        self.data_dir = data_dir
        self.model_name = model_name
        self.max_duration = max_duration
        self.sampling_rate = sampling_rate
        self.attack_type = attack_type
        self.attack_range = attack_range
        self.mode = mode
        self.max_files_per_class = max_files_per_class

        ai_files = get_sonics_file_paths(data_dir, model_name, "test")
        human_files = glob.glob(f"{data_dir}/real_songs/test/*.mp3", recursive=True)

        if self.max_files_per_class is not None:
            ai_files = ai_files[:self.max_files_per_class]
            human_files = human_files[:self.max_files_per_class]

        self.file_paths = []
        for files, label in [(ai_files, 1), (human_files, 0)]:
            for file in files:
                self.file_paths.append((file, label))

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path, label = self.file_paths[idx]
        waveform, sr = load_audio(file_path, self.max_duration)

        if self.attack_range and self.attack_type != "noattack":
            speed_factor = np.random.uniform(*self.attack_range)
        else:
            speed_factor = 1.0
        # apply attack if specified
        if self.attack_type == "resample":
            if self.mode == "continuous":
                waveform = resample(waveform, sr, speed_factor)
            elif self.mode == "discrete":
                pass  # Implement discrete speed factors if needed
        elif self.attack_type == "pitch_shift":
            if self.mode == "continuous":
                waveform = pitch_shift(waveform, sr, speed_factor)
            elif self.mode == "discrete":
                pass  # Implement discrete pitch shifts if needed

        target_sr = self.sampling_rate
        if sr != target_sr:
            resampled_waveform = soxr.resample(waveform.numpy().T, sr, target_sr, quality='VHQ').T
            waveform = torch.from_numpy(resampled_waveform)
            
        return waveform, label, speed_factor

In [3]:
import time
from tqdm.notebook import tqdm

test_dir = "/Users/emiledugelay/datasets/sonics/"
attack_range = [0.5, 2.0]
attack_type = "pitch_shift"  # "noattack", "resample", "pitch_shift"
mode = "continuous"
model_name = "udio_v120"  # "udio_v120", "suno_v3_5"

dataset = SonicsDataset(
    data_dir=test_dir,
    model_name=model_name,
    max_duration=120,
    sampling_rate=16000,
    attack_type=attack_type,
    attack_range=attack_range,
    mode=mode,
    max_files_per_class=1000,
)

results = []
for sample in tqdm(dataset):
    if sample is None:
        print("Skipping a sample due to loading error.")
        continue  # Skip samples that failed to load
    waveform, label, attack_factor = sample
    t0 = time.perf_counter()
    with torch.inference_mode():
        pred = model(waveform.to(device))
        pred = torch.sigmoid(pred).item()  # Convert to probability
    latency = (time.perf_counter() - t0) * 1000 # latency in milliseconds
    results.append((pred, label, attack_factor, latency))

  0%|          | 0/2000 [00:00<?, ?it/s]

In [6]:
import csv, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, precision_score, accuracy_score, recall_score, roc_curve, confusion_matrix

METRIC_RANGE = (0.7, 1.4)
AF_PLOT_POINTS = np.linspace(0.5, 2.0, 16)

def evaluate(results, attack_type="noattack", output_dir="results/spectttra_alpha/suno_v3_5/noattack"):
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    all_preds  = np.array([r[0] for r in results])
    all_labels = np.array([r[1] for r in results], dtype=int)
    all_afs    = np.array([r[2] for r in results])
    all_latencies = np.array([r[3] for r in results])

    latency_stats = {
        "latency_mean_ms": float(np.mean(all_latencies)),
        "throughput_samples_per_sec": float(len(all_latencies) / (np.sum(all_latencies) / 1000)),
    }

    lo, hi = METRIC_RANGE
    mask = (all_afs > lo) & (all_afs < hi)
    preds, labels = all_preds[mask], all_labels[mask]

    thresholds = np.linspace(0.01, 0.99, 199)
    f1s = [f1_score(labels, (preds >= t).astype(int), zero_division=0) for t in thresholds]
    best_t = float(thresholds[np.argmax(f1s)])
    pred_bin = (preds >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, pred_bin).ravel()

    metrics = {
        "auroc":     float(roc_auc_score(labels, preds)),
        "f1":        float(f1_score(labels, pred_bin, zero_division=0)),
        "precision": float(precision_score(labels, pred_bin, zero_division=0)),
        "accuracy":  float(accuracy_score(labels, pred_bin)),
        "recall":    float(recall_score(labels, pred_bin, zero_division=0)),
        "fpr":       float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
        "threshold": best_t,
    }

    stats = {**metrics, **latency_stats}
    name = f"sonics_test_{attack_type}"
    with open(out / f"{name}_metrics.json", "w") as f:
        json.dump(stats, f, indent=2)

    csv_path = out / "all_runs_metrics.csv"
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["run", *stats.keys()])
        if csv_path.stat().st_size == 0: w.writeheader()
        w.writerow({"run": name, **stats})

    _plot(preds, labels, metrics, best_t, f1s, thresholds, all_preds, all_labels, all_afs, out, name)
    print(f"Attacks in range {METRIC_RANGE}: ({mask.sum()}/{len(mask)}) | " + " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()))
    return stats

def _f1_vs_af(all_preds, all_labels, all_afs, best_t):
    f1s = np.full(len(AF_PLOT_POINTS), np.nan)
    for i, c in enumerate(AF_PLOT_POINTS):
        m = (all_afs >= c - 0.1) & (all_afs < c + 0.1)
        if m.sum() >= 2 and len(np.unique(all_labels[m])) == 2:
            f1s[i] = f1_score(all_labels[m], (all_preds[m] >= best_t).astype(int), zero_division=0)
    return f1s

def _plot(preds, labels, metrics, best_t, f1s, thresholds, all_preds, all_labels, all_afs, out, name):
    fig, axes = plt.subplots(1, 4, figsize=(25, 4))
    fig.suptitle("TEST | " + "  ".join(f"{k}={v:.3f}" for k, v in metrics.items() if k != "threshold"), fontsize=10)

    bins = np.linspace(0, 1, 51)
    axes[0].hist(preds[labels==0], bins=bins, alpha=0.7, density=True, label="Real")
    axes[0].hist(preds[labels==1], bins=bins, alpha=0.7, density=True, label="Fake")
    axes[0].axvline(best_t, color="yellow", linestyle="--", label=f"t={best_t:.2f}")
    axes[0].set(title="Score Distribution", xlabel="P(fake)"); axes[0].legend(fontsize=8)

    fpr, tpr, _ = roc_curve(labels, preds)
    axes[1].plot(fpr, tpr, label=f"AUC={metrics['auroc']:.3f}"); axes[1].plot([0,1],[0,1],"k--",linewidth=0.8)
    axes[1].set(title="ROC Curve", xlabel="FPR", ylabel="TPR"); axes[1].legend(fontsize=8)

    axes[2].plot(thresholds, f1s); axes[2].axvline(best_t, color="yellow", linestyle="--", label=f"best={best_t:.2f}")
    axes[2].set(title="F1 vs Threshold", xlabel="Threshold", ylabel="F1"); axes[2].legend(fontsize=8)

    f1_af = _f1_vs_af(all_preds, all_labels, all_afs, best_t)
    valid = ~np.isnan(f1_af)
    axes[3].plot(AF_PLOT_POINTS[valid], f1_af[valid], marker="o", linewidth=1.5, markersize=5)
    axes[3].axvspan(*METRIC_RANGE, alpha=0.12, color="green", label=f"metric range {METRIC_RANGE}")
    axes[3].set(title="F1 vs Attack Factor", xlabel="Attack Factor", ylabel="F1", xlim=(0.4, 2.1), ylim=(0, 1.05))
    axes[3].grid(alpha=0.3); axes[3].legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(out / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

In [7]:
evaluate(results, attack_type=attack_type, output_dir=f"results/spectttra_alpha/{model_name}/{attack_type}")

Attacks in range (0.7, 1.4): (910/2000) | auroc=0.7640 | f1=0.5565 | precision=0.8205 | accuracy=0.6637 | recall=0.4211 | fpr=0.0925 | threshold=0.0644


{'auroc': 0.763950073421439,
 'f1': 0.5565217391304348,
 'precision': 0.8205128205128205,
 'accuracy': 0.6637362637362637,
 'recall': 0.42105263157894735,
 'fpr': 0.09251101321585903,
 'threshold': 0.06444444444444444,
 'latency_mean_ms': 98.66221727873199,
 'throughput_samples_per_sec': 10.135592201165379}